# Resolution workflow
This is a prototype of a workflow to get resolution of an entire detector, both detector response and point spread function.
First a simple instrument with some additional monitors to produce some interesting data.

In [ ]:
import matplotlib.pyplot as plt
import mcstasscript as ms
import res_functions

instr = ms.McStas_instr("test")
source = instr.add_component("source", "Source_simple")
source.set_parameters(
    xwidth=0.1,
    yheight=0.1,
    focus_xw=0.02,
    focus_yh=0.03,
    dist=10,
    flux=1E15,
    E0=instr.add_parameter("E0", value=5.0, comment="source energy [meV]"),
    dE=instr.add_parameter("delta_E", value=0.1, comment="source energy spread [meV]"),
)

sample_position = instr.add_component("sample_position", "Arm")
sample_position.set_AT(source.dist, RELATIVE=source)

monitor = instr.add_component("before_sample", "Monitor_nD", RELATIVE=sample_position, AT=-0.1)
monitor.set_parameters(
    xwidth=0.1, yheight=0.1,
    filename='"before_sample.dat"',
    restore_neutron=1,
)
monitor.options = (
    f'"previous vx, vy, vz, neutron, list all neutrons"'
)

sample = instr.add_component("sample", "Res_sample", RELATIVE=sample_position)
sample.set_parameters(
    radius=0.01, yheight=0.03,
    target_x=0, target_y=0, target_z=1, focus_aw=2*175 + 1, focus_ah=30,
    E0=5, dE=instr.add_parameter("sample_dE", value=1, comment="sample energy spread [meV]"), elastic=0, delta_E_mode=1
)

monitor = instr.add_component("after_sample", "Monitor_nD", RELATIVE=sample_position)
monitor.set_parameters(
    filename='"after_sample.dat"',
    restore_neutron=1,
)
monitor.options = (
    f'"previous, x, y, z, vx, vy, vz, neutron, list all"'
)

detector_index = 0
pixel_min = 0

detector_direction = instr.add_component(
    "detector_direction_banana_1",
    "Arm",
    RELATIVE=sample_position,
    ROTATED=[0, 0, 0],
)

xbins = 20
ybins = 12
monitor = instr.add_component("Banana_1", "Monitor_nD")
monitor.set_parameters(
    radius=1.0,
    yheight=0.5,
    filename='"direct_event_banana_signal.dat"',
    restore_neutron=1,
)
monitor.options = (
    f'"banana theta bins={xbins} limits=[5, 175] '
    f'y bins={ybins}, neutron pixel min={pixel_min} t, list all neutrons"'
)

monitor.set_AT(0.0, RELATIVE=detector_direction)


instrument = instr

## Run the simulation
Now we run the simulation and extract the path to the file. It has to be done with NeXus

In [ ]:
instrument.settings(ncount=1E6, NeXus=True, suppress_output=True, mpi=6)
instrument.set_parameters(E0=6, delta_E=0.1, sample_dE=2)
instrument.show_parameters()

data = instrument.backengine()
file_path = data[0].original_data_location

### Correlate data
The event data is collected before and after the sample, as well as on the detector. We just need the data for rays that made it to the detector, this is sorted out by the function below.

In [ ]:
da = res_functions.reduced_scipp(file_path)
da

In [ ]:
# Check the limits
for variable in ["th", "t", "y", "sample_r", "delta_E"]:
    print(variable.ljust(20), str(da.coords[variable].min().value).ljust(30), str(da.coords[variable].max().value).ljust(20))

### Show widget
Here we show the widget with sliders that can cut along the data dimensions.
- small two theta, t, y sections results in detector response functions
- small qx, qy, qz and delta_E results in point spread functions

In [ ]:
%matplotlib widget
widget, subset_node = res_functions.make_widget(da, point_size=0.008)
widget

### Work with cut data
The cut data is available

In [ ]:
subset = subset_node()
subset

In [ ]:
subset.hist(qy=100, delta_E=100).plot()